In [1]:
!pip install spacy
!pip install pyspark
!pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425345 sha256=f9c62aed9ec8f3e60cca42c364564e54902ffbc07bf2bcc619bd17f2226a8944
  Stored in directory: /root/.cache/pip/wheels/41/4e/10/c2cf2467f71c678cfc8a6b9ac9241e5e44a01940da8fbb17fc
Successfully built pyspark


In [2]:
from nltk.corpus import stopwords
import spacy
import nltk

nltk.download('punkt')
nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words('english'))
stop_words = [i for i in stop_words if i not in ['not','until','against','up', 'down', 'no', 'nor',"aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn',
                                                "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't",
                                                 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't", "don't"]]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lower, regexp_replace, col, udf
from pyspark.ml.feature import StopWordsRemover
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from pyspark.sql.types import StringType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, Tokenizer, CountVectorizer
from pyspark.ml.feature import VectorAssembler

from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import Tokenizer, Word2Vec
from tensorflow.keras.layers import Dense, Flatten, Dropout, Embedding, GlobalMaxPooling1D, Conv1D, SimpleRNN, LSTM, GRU
from tensorflow.keras.optimizers import Adam
from pyspark.sql.functions import split
from pyspark.sql.functions import size
from pyspark.sql.functions import when

import numpy as np
import pandas as pd

In [28]:
%%time
spark = SparkSession.builder.appName("SparkByExample").getOrCreate()


tweet_ = spark.read.csv("/content/Sentiment_tweet4.csv", inferSchema=True, header=True)

tweet_ = tweet_.toDF(*["_c0", "label", "id", "Time", "States", "Name", "Tweet"])
tweet_ = tweet_.drop(*["_c0","Time", "id", "States", "Name"])


# tweet_ = tweet_.orderBy(rand()) # Shuffle

def preprocess_tweet(tweet_):
    tweet_ = lower(tweet_)
    tweet_ = regexp_replace(tweet_, r"http\S+|www\S+|https\S+", "")
    tweet_ = regexp_replace(tweet_, r"@\w+", "")
    tweet_ = regexp_replace(tweet_, r"[^\w\s#]", "")

    return tweet_


def tokenization(tweet_):
    tokens = [token.lemma_ for token in nlp(tweet_)]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [token for token in tokens if len(token) > 1]
    processed_tweet = ' '.join(tokens)
    return processed_tweet


Tokenize_ = udf(lambda x:tokenization(x),StringType())

tweet_ = tweet_.withColumn("Cleaned_data", preprocess_tweet(col("Tweet")))
tweet_ = tweet_.withColumn("Cleaned_data", Tokenize_(col("Cleaned_data")))

CPU times: user 296 ms, sys: 76.8 ms, total: 373 ms
Wall time: 1.47 s


In [6]:
%%time
tweet_.show()

+-----+--------------------+--------------------+
|label|               Tweet|        Cleaned_data|
+-----+--------------------+--------------------+
|    0|internet is slow ...|internet slow tod...|
|    4|@tooclevername to...|hammer everything...|
|    0|These kids are cr...|      kid crazy save|
|    0|my puppy libby sc...|puppy libby scrat...|
|    4|okay, my ipod has...|okay ipod sync fi...|
|    4|Good morning twit...|good morning twit...|
|    0|Fed looks a bit s...|fed look bit shak...|
|    4|@rahulnag congo!!...|congo get mine ar...|
|    4|G'Morning Twitter...|gmorne twittervil...|
|    0|@marrzbarz stop s...|stop sleep late c...|
|    0|Feeling down :\ G...|feel down go slee...|
|    0| my eva got hit b...|eva got hit drunk...|
|    0|@aurorawilbs at l...|least ur sun rebo...|
|    0|@BenJoBubble hell...|hello mate favor ...|
|    4|@tb78 Cheers for ...|        cheer follow|
|    4|WOOHOO today is F...|woohoo today frid...|
|    0|Washed all my bru...|wash brush last n...|


In [29]:
%%time
tweet = tweet_.withColumn("Cleaned_data", split(tweet_["Cleaned_data"], " "))
tweet = tweet.select("Cleaned_data", "label")

CPU times: user 6.2 ms, sys: 24 µs, total: 6.22 ms
Wall time: 36 ms


In [30]:
tweet.count() #Len

2000

# Spark Neural Network + CountVectorize

In [31]:
%%time
cv = CountVectorizer(inputCol="Cleaned_data", outputCol="features")
cvModel = cv.fit(tweet)
vectorizedData = cvModel.transform(tweet)


vectorizedData = vectorizedData.withColumn("label", when(vectorizedData["label"] == 4, 1).otherwise(0))
# Convert labels to numerical indices
indexer = StringIndexer(inputCol="label", outputCol="indexedLabel")
indexedData = indexer.fit(vectorizedData).transform(vectorizedData)

# Split data into training and test sets
(trainingData, testData) = indexedData.randomSplit([0.8, 0.2])

# Define the layers for the neural network
layers = [len(cvModel.vocabulary), 64, 32, 2]  # Change units as needed

# Create the Multilayer Perceptron Classifier
mlp = MultilayerPerceptronClassifier(layers=layers, seed=1234, featuresCol='features', labelCol='label',
                                     blockSize=128, solver='l-bfgs', maxIter=100, tol=1e-6)



# Assuming 'label' is the column containing your labels


# Fit the model
model = mlp.fit(trainingData)

# Make predictions
predictions = model.transform(testData)

# Evaluate the model
evaluator = MulticlassClassificationEvaluator(metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Test set accuracy = " + str(accuracy))


Test set accuracy = 0.6470588235294118
CPU times: user 1 s, sys: 120 ms, total: 1.12 s
Wall time: 2min 47s


In [32]:
%%time
predictions = model.transform(trainingData)

# Evaluate the model
evaluator = MulticlassClassificationEvaluator(metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Train set accuracy = " + str(accuracy))

Train set accuracy = 0.997513983840895
CPU times: user 154 ms, sys: 14.5 ms, total: 169 ms
Wall time: 23.2 s


# Spark Neural Network + Word2Vec

In [33]:
%%time
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, Word2Vec


tweet_array_ = tweet.withColumn("label", when(tweet["label"] == 4, 1).otherwise(0))

word2Vec = Word2Vec(vectorSize=100, minCount=5, inputCol="Cleaned_data", outputCol="word2vec")
word2VecModel = word2Vec.fit(tweet_array_)
word2vecData = word2VecModel.transform(tweet_array_)

vectorAssembler = VectorAssembler(inputCols=["word2vec"], outputCol="features")
assembledData = vectorAssembler.transform(word2vecData)



(trainingData, testData) = tweet_array_.randomSplit([0.8, 0.2], seed=42)


layers = [100, 64, 32, 2]


mlp = MultilayerPerceptronClassifier(layers=layers, seed=1234, featuresCol='features', labelCol='label',
                                     blockSize=128, solver='l-bfgs', maxIter=100, tol=1e-6)

pipeline = Pipeline(stages=[word2Vec,vectorAssembler, mlp])


model = pipeline.fit(trainingData)


predictions = model.transform(testData)


evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Test set accuracy = {accuracy}")


Test set accuracy = 0.5614525139664804
CPU times: user 912 ms, sys: 119 ms, total: 1.03 s
Wall time: 2min 25s


In [34]:
%%time
predictions = model.transform(trainingData)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Train set accuracy = {accuracy}")

Train set accuracy = 0.5481120584652862
CPU times: user 137 ms, sys: 26 ms, total: 163 ms
Wall time: 21 s


# CountVectorize + Dense Model

In [36]:
%%time
embeddings = np.array(vectorizedData.select('features').rdd.map(lambda x: np.array(x[0].toArray())).collect())
labels = np.array(vectorizedData.select('label').rdd.map(lambda x: x[0]).collect())

embeddings_tensor = tf.convert_to_tensor(embeddings, dtype=tf.float32)
labels_tensor = tf.convert_to_tensor(labels, dtype=tf.float32)

model = Sequential()
model.add(Dense(2048, activation='relu', input_shape=(embeddings.shape[1],)))
model.add(Dense(512, activation='relu'))
model.add(Dropout(.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(.5))
model.add(Dense(32, activation='relu'))
model.add(Dropout(.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer = Adam(learning_rate = 0.0001), metrics=['accuracy'])
model.fit(embeddings_tensor, labels_tensor, epochs=10, batch_size=32, validation_split=0.2)

print('\n')
print("Model Evaluation on Data : ", round(model.evaluate(embeddings_tensor, labels_tensor)[1]*100,2))

Epoch 1/10
50/50 [==============================] - 12s 225ms/step - loss: 0.6925 - accuracy: 0.5156 - val_loss: 0.6912 - val_accuracy: 0.5375
Epoch 2/10
50/50 [==============================] - 11s 220ms/step - loss: 0.6871 - accuracy: 0.5587 - val_loss: 0.6889 - val_accuracy: 0.5675
Epoch 3/10
50/50 [==============================] - 10s 196ms/step - loss: 0.6777 - accuracy: 0.5869 - val_loss: 0.6808 - val_accuracy: 0.5950
Epoch 4/10
50/50 [==============================] - 10s 208ms/step - loss: 0.6473 - accuracy: 0.6787 - val_loss: 0.6647 - val_accuracy: 0.6325
Epoch 5/10
50/50 [==============================] - 11s 222ms/step - loss: 0.6086 - accuracy: 0.7287 - val_loss: 0.6407 - val_accuracy: 0.6425
Epoch 6/10
50/50 [==============================] - 11s 227ms/step - loss: 0.5118 - accuracy: 0.8300 - val_loss: 0.6173 - val_accuracy: 0.6800
Epoch 7/10
50/50 [==============================] - 12s 233ms/step - loss: 0.3725 - accuracy: 0.8925 - val_loss: 0.6395 - val_accuracy: 0.6475

#LSTM + Word2Vec

In [43]:
%%time
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

embeddings = np.array(assembledData.select('word2vec').rdd.map(lambda x: np.array(x[0].toArray())).collect())
labels = np.array(assembledData.select('label').rdd.map(lambda x: x[0]).collect())

model = Sequential()

model.add(Embedding(max_words, 128, input_length=max_sequence_length))
model.add(LSTM(units=128, dropout=0.2, return_sequences=True))
model.add(LSTM(64))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer = Adam(learning_rate = 0.0001), metrics=['accuracy'])
model.fit(embeddings, labels,  epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
50/50 [==============================] - 21s 309ms/step - loss: 0.6928 - accuracy: 0.5056 - val_loss: 0.6936 - val_accuracy: 0.4800
Epoch 2/10
50/50 [==============================] - 14s 284ms/step - loss: 0.6929 - accuracy: 0.5044 - val_loss: 0.6939 - val_accuracy: 0.4800
Epoch 3/10
50/50 [==============================] - 15s 302ms/step - loss: 0.6936 - accuracy: 0.4938 - val_loss: 0.6937 - val_accuracy: 0.4800
Epoch 4/10
50/50 [==============================] - 15s 296ms/step - loss: 0.6931 - accuracy: 0.5000 - val_loss: 0.6935 - val_accuracy: 0.4800
Epoch 5/10
50/50 [==============================] - 15s 296ms/step - loss: 0.6929 - accuracy: 0.5038 - val_loss: 0.6934 - val_accuracy: 0.4800
Epoch 6/10
50/50 [==============================] - 15s 311ms/step - loss: 0.6937 - accuracy: 0.4925 - val_loss: 0.6935 - val_accuracy: 0.4800
Epoch 7/10
50/50 [==============================] - 15s 308ms/step - loss: 0.6930 - accuracy: 0.5194 - val_loss: 0.6935 - val_accuracy: 0.4800

# CNN + Word2Vec

In [38]:
%%time
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

embeddings = np.array(assembledData.select('word2vec').rdd.map(lambda x: np.array(x[0].toArray())).collect())
labels = np.array(assembledData.select('label').rdd.map(lambda x: x[0]).collect())


embedding_dim = 100
sequence_length = 100
vocab_size = 10000

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=sequence_length))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dropout(0.3))
model.add(Dense(units=64, activation='relu'))
model.add(Dense(units=1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(embeddings, labels, epochs=10, batch_size=32, validation_split=0.2)

print('\n')
print("Model Evaluation on Data : ", round(model.evaluate(embeddings, labels)[1]*100,2))

Epoch 1/10
50/50 [==============================] - 4s 65ms/step - loss: 0.6939 - accuracy: 0.4944 - val_loss: 0.6930 - val_accuracy: 0.5200
Epoch 2/10
50/50 [==============================] - 3s 61ms/step - loss: 0.6945 - accuracy: 0.5194 - val_loss: 0.6976 - val_accuracy: 0.4800
Epoch 3/10
50/50 [==============================] - 2s 43ms/step - loss: 0.6944 - accuracy: 0.4919 - val_loss: 0.6927 - val_accuracy: 0.5200
Epoch 4/10
50/50 [==============================] - 2s 42ms/step - loss: 0.6935 - accuracy: 0.4856 - val_loss: 0.6933 - val_accuracy: 0.4800
Epoch 5/10
50/50 [==============================] - 2s 43ms/step - loss: 0.6932 - accuracy: 0.4969 - val_loss: 0.6934 - val_accuracy: 0.4800
Epoch 6/10
50/50 [==============================] - 2s 42ms/step - loss: 0.6937 - accuracy: 0.5088 - val_loss: 0.6934 - val_accuracy: 0.4800
Epoch 7/10
50/50 [==============================] - 3s 62ms/step - loss: 0.6935 - accuracy: 0.5038 - val_loss: 0.6944 - val_accuracy: 0.4800
Epoch 8/10
50

also there is another model with 79 accuracy. if you want to check this you can only visit https://www.kaggle.com/code/ashishjangra27/word-embedding-techniques-in-nlp.

Epoch 1/12
313/313 [==============================] - 496s 2s/step - loss: 0.5333 - accuracy: 0.7381 - val_loss: 0.4733 - val_accuracy: 0.7742

Epoch 2/12
313/313 [==============================] - 474s 2s/step - loss: 0.4773 - accuracy: 0.7703 - val_loss: 0.4615 - val_accuracy: 0.7810

Epoch 3/12
313/313 [==============================] - 472s 2s/step - loss: 0.4670 - accuracy: 0.7766 - val_loss: 0.4547 - val_accuracy: 0.7851

Epoch 4/12
313/313 [==============================] - 480s 2s/step - loss: 0.4602 - accuracy: 0.7813 - val_loss: 0.4498 - val_accuracy: 0.7876

Epoch 5/12
313/313 [==============================] - 473s 2s/step - loss: 0.4551 - accuracy: 0.7838 - val_loss: 0.4470 - val_accuracy: 0.7893

Epoch 6/12
313/313 [==============================] - 471s 2s/step - loss: 0.4511 - accuracy: 0.7863 - val_loss: 0.4433 - val_accuracy: 0.7915

Epoch 7/12
313/313 [==============================] - 473s 2s/step - loss: 0.4471 - accuracy: 0.7889 - val_loss: 0.4423 - val_accuracy: 0.7920

Epoch 8/12
313/313 [==============================] - 473s 2s/step - loss: 0.4443 - accuracy: 0.7904 - val_loss: 0.4380 - val_accuracy: 0.7947

Epoch 9/12
313/313 [==============================] - 474s 2s/step - loss: 0.4413 - accuracy: 0.7922 - val_loss: 0.4368 - val_accuracy: 0.7949

Epoch 10/12
313/313 [==============================] - 474s 2s/step - loss: 0.4394 - accuracy: 0.7929 - val_loss: 0.4342 - val_accuracy: 0.7969

Epoch 11/12
313/313 [==============================] - 476s 2s/step - loss: 0.4362 - accuracy: 0.7952 - val_loss: 0.4336 - val_accuracy: 0.7970

Epoch 12/12
313/313 [==============================] - 476s 2s/step - loss: 0.4347 - accuracy: 0.7961 - val_loss: 0.4318 - val_accuracy: 0.7987


20000/20000 [==============================] - 500s 25ms/step - loss: 0.4317 - accuracy: 0.7986
Model Evaluation on Testing Data :  79.86